# Plot Light Curves vs MJD for Stable Stars

For each stable star in `data_MergeVisits_02_out/per_star/`, this notebook
produces a **GridSpec 6×2 figure** per star:

- **Left column (6 rows)** — `psfFlux` vs `expMidptMJD` per band (order: u g r i z y).  
  A secondary top x-axis shows the calendar date (YYYY-MM-DD).  
  The y-axis is clipped to ±N·σ_IQR around the median to make scatter vs.
  error bars immediately visible.  
  Individual error bars (`psfFluxErr`) are drawn; the Gaussian fit mean is
  shown as a horizontal dashed line and the ±σ_IQR band as a shaded region.

- **Right column (6 rows)** — histogram of `psfFlux` for the same band with a
  Gaussian fit overlaid; mean (μ) and σ from the fit are annotated.

Bands with fewer than 3 good measurements are skipped (empty subplot).

**Section 8** produces an enriched precision summary table per star × band
(RMS, σ_fit, σ_IQR, scatter in % and mmag).

**Section 9** produces a stacked 2×3 panel figure showing the normalised
flux residuals `(flux - median) / median` across all stars and all bands,
with a per-star colour coding and a global Gaussian fit overlay.

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-27
- **Last update:** 2026-06-29  *(patched: precision table + stacked residuals figure)*


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator

from astropy.time import Time
from scipy.stats import norm
from scipy.optimize import curve_fit

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)
log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ──────────────────────────────────────────────────────
NB_TAG = "PlotLC_03"

# ── Input: merged LC files from notebook 02 ───────────────────────────────
DIR_DATA_IN = "./data_MergeVisits_02_out"
DIR_PER_STAR_IN = os.path.join(DIR_DATA_IN, "per_star")

# ── Output figures ────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# ── Photometric columns ───────────────────────────────────────────────
MJD_COL = "expMidptMJD"
FLUX_COL = "psfFlux"
FLUX_ERR_COL = "psfFluxErr"
BAND_COL = "band"

# ── Band ordering and colours (LSST ugrizy) ───────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#6C2DC7",  # violet
    "g": "#1CB94A",  # green
    "r": "#E8262A",  # red
    "i": "#E87E26",  # orange
    "z": "#C95C93",  # pink
    "y": "#994D00",  # brown
}

# ── Y-axis clipping for the light-curve panels ────────────────────────────
# The y-axis is set to  median ± N_SIGMA_YLIM × σ_IQR  where
#   σ_IQR = IQR / 1.349  (robust estimator of the standard deviation)
# Increase N_SIGMA_YLIM to see more outliers; decrease to zoom in.
N_SIGMA_YLIM = 5.0

# ── Minimum number of good points to attempt a Gaussian fit ─────────────────
MIN_POINTS = 5

# ── Figure size (width, height) in inches ─────────────────────────────────
FIG_WIDTH = 16
FIG_HEIGHT = 10

## 4. Helper functions

In [ ]:
# ── MJD → datetime (for the secondary top x-axis) ───────────────────────
def mjd_to_datetime(mjd_array):
    """Convert an array of MJD floats to numpy datetime64 objects."""
    return Time(mjd_array, format="mjd").to_datetime()


# ── Robust σ from IQR ────────────────────────────────────────────────
def sigma_iqr(values):
    """Robust standard-deviation estimate via the inter-quartile range.
    σ_IQR = IQR / 1.3489795
    """
    q75, q25 = np.nanpercentile(values, [75, 25])
    return (q75 - q25) / 1.3489795


# ── Gaussian model for curve_fit ─────────────────────────────────────
def gaussian(x, mu, sigma, amplitude):
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


# ── Fit a Gaussian to a histogram ──────────────────────────────────────
def fit_gaussian_to_flux(flux_values, n_bins=30, flux_range=None):
    """Fit a Gaussian to a histogram of *flux_values*.

    Parameters
    ----------
    flux_values : array-like
        psfFlux measurements for one band.
    n_bins : int
        Number of histogram bins.
    flux_range : (float, float) or None
        If provided, the histogram is built only over this [lo, hi] window,
        so that the histogram is centred on the median and consistent with
        the y-axis range of the light-curve panel.
        If None, the full data range is used (old behaviour).

    Returns
    -------
    (mu, sigma, amplitude, bin_centers, hist_counts) or
    (None, None, None, bin_centers, hist_counts) on failure.
    """
    # Build histogram over the requested window; clip outliers outside it
    hist_kw = {"bins": n_bins}
    if flux_range is not None:
        lo, hi = flux_range
        hist_kw["range"] = (lo, hi)

    counts, bin_edges = np.histogram(flux_values, **hist_kw)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    mu0 = np.median(flux_values)
    sigma0 = sigma_iqr(flux_values)
    amp0 = counts.max()

    if sigma0 <= 0 or amp0 <= 0:
        return None, None, None, bin_centers, counts

    try:
        popt, _ = curve_fit(
            gaussian,
            bin_centers,
            counts,
            p0=[mu0, sigma0, amp0],
            maxfev=5000,
        )
        mu_fit, sigma_fit, amp_fit = popt
        # Reject unphysical fits
        if sigma_fit <= 0 or abs(mu_fit - mu0) > 5 * sigma0:
            return None, None, None, bin_centers, counts
        return abs(mu_fit), abs(sigma_fit), abs(amp_fit), bin_centers, counts
    except RuntimeError:
        return None, None, None, bin_centers, counts


# ── savefig: PDF + PNG ───────────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)


# ── flux scatter in mmag ─────────────────────────────────────────────
def sigma_nJy_to_mmag(sigma_nJy, median_nJy):
    """Convert a flux scatter sigma (nJy) to mmag using the linear approximation:
       delta_mag ≃ (2.5 / ln10) * (sigma / median)  [mag]
    Returns the value in milli-magnitudes.
    The sign convention is always positive.
    """
    if median_nJy <= 0 or sigma_nJy < 0:
        return np.nan
    return 1000.0 * (2.5 / np.log(10)) * (sigma_nJy / abs(median_nJy))

## 5. Core plotting function: one star → one 6×2 GridSpec figure

In [ ]:
def plot_star_lc(df_star: pd.DataFrame, simbad_id: str, file_stem: str) -> None:
    """Build and save the 6x2 GridSpec figure for one stable star.

    Parameters
    ----------
    df_star    : merged LC DataFrame for this star.
    simbad_id  : human-readable identifier shown in the figure title.
    file_stem  : output filename stem (no extension, no directory).
    """
    # Drop rows with NaN flux or MJD
    df = df_star.dropna(subset=[MJD_COL, FLUX_COL]).copy()

    # ── Global MJD range (shared x-axis for all band panels) ──────────
    # Compute once from ALL valid rows so every band panel (u g r i z y)
    # uses the exact same MJDMIN / MJDMAX on its x-axis.
    valid_mjd_mask = df[MJD_COL].notna()
    if FLUX_ERR_COL in df.columns:
        valid_mjd_mask &= df[FLUX_ERR_COL].notna() & (df[FLUX_ERR_COL] > 0)
    valid_mjd_all = df.loc[valid_mjd_mask, MJD_COL]

    if len(valid_mjd_all) > 0:
        global_mjd_min = valid_mjd_all.min()
        global_mjd_max = valid_mjd_all.max()
    else:
        global_mjd_min, global_mjd_max = 0.0, 1.0  # fallback

    # Add a small padding (0.5% of range, min 0.5 day) so points near
    # the edges are not clipped by the axis frame.
    mjd_pad = max((global_mjd_max - global_mjd_min) * 0.005, 0.5)
    global_mjd_min -= mjd_pad
    global_mjd_max += mjd_pad

    # Pre-compute date tick positions once from the global range
    global_mjd_range = global_mjd_max - global_mjd_min
    if global_mjd_range < 1:
        n_ticks = 4
    else:
        n_ticks = min(5, max(2, int(global_mjd_range // 30) + 2))
    global_tick_mjd = np.linspace(global_mjd_min, global_mjd_max, n_ticks)
    global_tick_dates = Time(global_tick_mjd, format="mjd").to_value("iso", subfmt="date")

    # ── Figure & GridSpec ──────────────────────────────────────────────
    fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
    # Show the common MJD range in the suptitle for easy verification
    fig.suptitle(
        f"{simbad_id}\n"
        f"psfFlux light curves per band   |   y-axis: median +/- {N_SIGMA_YLIM:.0f}*sigma_IQR\n"
        f"Common MJD range: {global_mjd_min + mjd_pad:.2f} -- {global_mjd_max - mjd_pad:.2f}",
        fontsize=13,
        fontweight="bold",
        y=0.995,
    )

    # 6 rows x 2 columns; left panels wider than right
    gs = gridspec.GridSpec(
        6,
        2,
        figure=fig,
        width_ratios=[3, 1],
        hspace=0.55,
        wspace=0.28,
    )

    for row_idx, band in enumerate(BAND_ORDER):
        color = BAND_COLORS.get(band, "steelblue")

        ax_lc = fig.add_subplot(gs[row_idx, 0])  # left: light curve
        ax_hist = fig.add_subplot(gs[row_idx, 1])  # right: histogram

        # Select this band, require valid flux and error
        mask = (df[BAND_COL] == band) & df[MJD_COL].notna() & df[FLUX_COL].notna()
        # Also exclude flagged rows if psfFluxErr is 0 or NaN
        if FLUX_ERR_COL in df.columns:
            mask &= df[FLUX_ERR_COL].notna() & (df[FLUX_ERR_COL] > 0)

        grp = df[mask].sort_values(MJD_COL)
        n_pts = len(grp)

        # ── Empty band subplot ───────────────────────────────────────────
        if n_pts < MIN_POINTS:
            for ax in (ax_lc, ax_hist):
                ax.set_visible(True)
                ax.text(
                    0.5,
                    0.5,
                    f"{band}  --  {n_pts} point(s)",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="grey",
                    fontsize=10,
                )
                ax.set_xticks([])
                ax.set_yticks([])
            # Still enforce common MJD range even on empty LC panel
            ax_lc.set_xlim(global_mjd_min, global_mjd_max)
            continue

        mjd = grp[MJD_COL].values
        flux = grp[FLUX_COL].values
        err = grp[FLUX_ERR_COL].values if FLUX_ERR_COL in grp.columns else np.zeros_like(flux)

        # ── Robust statistics ──────────────────────────────────────────
        med = np.nanmedian(flux)
        sig_iqr = sigma_iqr(flux)
        if sig_iqr == 0:
            sig_iqr = np.nanstd(flux) or 1.0

        # ── Gaussian fit centred on median +/- N_SIGMA_YLIM * sigma_IQR ─
        hist_lo = med - N_SIGMA_YLIM * sig_iqr
        hist_hi = med + N_SIGMA_YLIM * sig_iqr
        mu_fit, sigma_fit, amp_fit, bin_centers, hist_counts = fit_gaussian_to_flux(
            flux, flux_range=(hist_lo, hist_hi)
        )

        # ── LEFT panel: light curve ────────────────────────────────────────
        ax_lc.errorbar(
            mjd,
            flux,
            yerr=err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            alpha=0.8,
            elinewidth=0.8,
            capsize=2,
            label=f"{band}  (N={n_pts})",
            zorder=3,
        )

        # Median line
        ax_lc.axhline(med, color=color, lw=1.4, ls="--", alpha=0.9, zorder=2)

        # +/- sigma_IQR shaded band
        ax_lc.axhspan(
            med - sig_iqr,
            med + sig_iqr,
            color=color,
            alpha=0.10,
            zorder=1,
        )

        # y-axis limits: median +/- N_SIGMA_YLIM * sigma_IQR
        ylim_lo = med - N_SIGMA_YLIM * sig_iqr
        ylim_hi = med + N_SIGMA_YLIM * sig_iqr
        # Safety: never collapse to zero range
        if ylim_hi - ylim_lo < 1e-10 * abs(med) + 1.0:
            ylim_lo = med - 1.0
            ylim_hi = med + 1.0
        ax_lc.set_ylim(ylim_lo, ylim_hi)

        # ── Enforce common MJD x-axis for ALL band panels ─────────────────
        ax_lc.set_xlim(global_mjd_min, global_mjd_max)

        # Annotation: sigma_IQR / median  (relative scatter in %)
        rel_scatter = 100.0 * sig_iqr / abs(med) if abs(med) > 0 else np.nan
        ax_lc.text(
            0.02,
            0.96,
            f"sigma_IQR = {sig_iqr:.1f} nJy   ({rel_scatter:.2f}%)",
            transform=ax_lc.transAxes,
            fontsize=8,
            va="top",
            ha="left",
            color=color,
        )

        ax_lc.set_ylabel(f"{band}  psfFlux [nJy]", fontsize=9, color=color)
        ax_lc.tick_params(axis="y", labelsize=8)
        ax_lc.yaxis.set_minor_locator(AutoMinorLocator())
        ax_lc.tick_params(which="minor", length=2)

        # x-axis: MJD ticks on the bottom, date labels on top
        ax_lc.tick_params(axis="x", labelsize=7)

        # Secondary top x-axis with calendar dates using the global range
        ax_top = ax_lc.twiny()
        ax_top.set_xlim(global_mjd_min, global_mjd_max)  # same global range
        ax_top.set_xticks(global_tick_mjd)
        ax_top.set_xticklabels(global_tick_dates, fontsize=7, rotation=20, ha="left")
        ax_top.tick_params(length=3)

        if row_idx == len(BAND_ORDER) - 1:
            ax_lc.set_xlabel("expMidptMJD", fontsize=9)

        ax_lc.legend(loc="upper right", fontsize=8, framealpha=0.6)
        ax_lc.grid(True, lw=0.4, alpha=0.4)

        # ── RIGHT panel: histogram + Gaussian fit ────────────────────────────
        ax_hist.bar(
            bin_centers,
            hist_counts,
            width=(bin_centers[1] - bin_centers[0]) * 0.9,
            color=color,
            alpha=0.55,
            edgecolor="none",
            label="data",
        )

        if mu_fit is not None:
            x_fit = np.linspace(bin_centers[0], bin_centers[-1], 300)
            y_fit = gaussian(x_fit, mu_fit, sigma_fit, amp_fit)
            ax_hist.plot(x_fit, y_fit, color="k", lw=1.5, label="Gauss fit")
            ax_hist.axvline(mu_fit, color="k", lw=1.2, ls="--")
            ax_hist.axvline(mu_fit - sigma_fit, color="grey", lw=0.9, ls=":")
            ax_hist.axvline(mu_fit + sigma_fit, color="grey", lw=0.9, ls=":")
            ax_hist.text(
                0.97,
                0.96,
                f"mu = {mu_fit:.1f}\nsigma = {sigma_fit:.1f}",
                transform=ax_hist.transAxes,
                fontsize=8,
                va="top",
                ha="right",
                color="k",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
            )
        else:
            ax_hist.text(
                0.97,
                0.96,
                "fit failed",
                transform=ax_hist.transAxes,
                fontsize=8,
                va="top",
                ha="right",
                color="red",
            )

        ax_hist.set_xlabel("psfFlux [nJy]", fontsize=8)
        ax_hist.set_ylabel("counts", fontsize=8)
        ax_hist.tick_params(labelsize=8)
        ax_hist.legend(fontsize=7, loc="upper left")
        ax_hist.set_xlim(hist_lo, hist_hi)
        ax_hist.set_title(f"{band} band", fontsize=9, color=color)
        ax_hist.grid(True, lw=0.3, alpha=0.35)

    savefig(fig, file_stem)
    # plt.close(fig)
    gc.collect()

## 6. Discover per-star files and run the plotting loop

In [ ]:
lc_files = sorted(f for f in os.listdir(DIR_PER_STAR_IN) if f.endswith("_lc_mjd.csv"))
log.info("Found %d per-star LC files.", len(lc_files))
lc_files

In [ ]:
n_ok = 0
n_err = 0

for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)

    try:
        df_star = pd.read_csv(src_path)
    except Exception as exc:
        log.error("  ERROR reading %s: %s", fname, exc)
        n_err += 1
        continue

    # simbad_id from the first valid row
    simbad_id = (
        df_star["simbad_id"].iloc[0]
        if "simbad_id" in df_star.columns and len(df_star) > 0
        else fname.replace("_lc_mjd.csv", "")
    )

    # Output figure stem: strip the suffix
    file_stem = fname.replace("_lc_mjd.csv", "") + "_LC_MJD"

    log.info("Plotting: %s  (%d rows)", simbad_id, len(df_star))

    try:
        plot_star_lc(df_star, simbad_id, file_stem)
        n_ok += 1
    except Exception as exc:
        log.error("  ERROR plotting %s: %s", simbad_id, exc)
        plt.close("all")
        n_err += 1

log.info("Done — %d figures saved, %d errors.", n_ok, n_err)

## 7. Quick inline preview of one figure

In [ ]:
# Display the first saved PNG inline for a quick check
from IPython.display import Image, display

saved_pngs = sorted(os.path.join(DIR_FIGS, f) for f in os.listdir(DIR_FIGS) if f.endswith(".png"))

if saved_pngs:
    log.info("Previewing: %s", saved_pngs[0])
    display(Image(filename=saved_pngs[0], width=900))
else:
    log.warning("No PNG figure found in %s", DIR_FIGS)

## 8. Precision summary table: RMS, σ_fit, σ_IQR, scatter in % and mmag

For each (star × band) combination with at least `MIN_POINTS` good measurements,
we compute:

| column | definition |
|--------|------------|
| `N` | number of valid measurements |
| `median_flux_nJy` | robust central value |
| `RMS_nJy` | √⟨(flux − median)²⟩ (classical RMS around median) |
| `sigma_fit_nJy` | σ of Gaussian fit to flux histogram |
| `sigma_IQR_nJy` | IQR/1.349 (robust σ) |
| `median_err_nJy` | median of individual `psfFluxErr` |
| `RMS_%` | 100 × RMS / median |
| `sigma_fit_%` | 100 × σ_fit / median |
| `sigma_IQR_%` | 100 × σ_IQR / median |
| `RMS_mmag` | RMS converted to milli-magnitudes |
| `sigma_fit_mmag` | σ_fit in milli-magnitudes |
| `sigma_IQR_mmag` | σ_IQR in milli-magnitudes |
| `scatter/noise` | σ_IQR / median(psfFluxErr) |
| `excess_scatter` | True if scatter/noise > 2 |

In [ ]:
# ── Load the global merged file ───────────────────────────────────────
global_csv = os.path.join(DIR_DATA_IN, "all_stars_lightcurves_mjd.csv")
df_all = pd.read_csv(global_csv)
df_all = df_all.dropna(subset=[MJD_COL, FLUX_COL])
if FLUX_ERR_COL in df_all.columns:
    df_all = df_all[df_all[FLUX_ERR_COL] > 0]

rows = []
for (sid, band), grp in df_all.groupby(["simbad_id", BAND_COL]):
    flux = grp[FLUX_COL].values
    err = grp[FLUX_ERR_COL].values if FLUX_ERR_COL in grp.columns else np.full_like(flux, np.nan)

    if len(flux) < MIN_POINTS:
        continue

    med = np.nanmedian(flux)
    rms = np.sqrt(np.nanmean((flux - med) ** 2))
    sig_iqr_ = sigma_iqr(flux)
    med_err = np.nanmedian(err)

    # Gaussian fit on the flux histogram (window = median ± 5*sigma_IQR)
    sig_iqr_safe = sig_iqr_ if sig_iqr_ > 0 else (np.nanstd(flux) or 1.0)
    flo = med - N_SIGMA_YLIM * sig_iqr_safe
    fhi = med + N_SIGMA_YLIM * sig_iqr_safe
    mu_fit, sigma_fit, _, _, _ = fit_gaussian_to_flux(flux, flux_range=(flo, fhi))
    sigma_fit = sigma_fit if sigma_fit is not None else np.nan

    rows.append(
        {
            "simbad_id": sid,
            "band": band,
            "N": len(flux),
            "median_flux_nJy": round(med, 2),
            "RMS_nJy": round(rms, 2),
            "sigma_fit_nJy": round(sigma_fit, 2) if np.isfinite(sigma_fit) else np.nan,
            "sigma_IQR_nJy": round(sig_iqr_, 2),
            "median_err_nJy": round(med_err, 2),
            "RMS_%": round(100.0 * rms / abs(med), 4) if abs(med) > 0 else np.nan,
            "sigma_fit_%": round(100.0 * sigma_fit / abs(med), 4)
            if (np.isfinite(sigma_fit) and abs(med) > 0)
            else np.nan,
            "sigma_IQR_%": round(100.0 * sig_iqr_ / abs(med), 4) if abs(med) > 0 else np.nan,
            "RMS_mmag": round(sigma_nJy_to_mmag(rms, med), 2),
            "sigma_fit_mmag": round(sigma_nJy_to_mmag(sigma_fit, med), 2)
            if np.isfinite(sigma_fit)
            else np.nan,
            "sigma_IQR_mmag": round(sigma_nJy_to_mmag(sig_iqr_, med), 2),
            "scatter/noise": round(sig_iqr_ / med_err, 3) if med_err > 0 else np.nan,
            "excess_scatter": bool(sig_iqr_ / med_err > 2.0) if med_err > 0 else False,
        }
    )

# ── Sort by star name and band order ─────────────────────────────────
df_precision = pd.DataFrame(rows)
band_order_map = {b: i for i, b in enumerate(BAND_ORDER)}
df_precision["_band_ord"] = df_precision["band"].map(band_order_map)
df_precision = df_precision.sort_values(["simbad_id", "_band_ord"]).drop(columns=["_band_ord"])
df_precision = df_precision.reset_index(drop=True)

print(f"Precision summary — {len(df_precision)} (star × band) combinations")
df_precision

In [ ]:
# ── Styled display: colour-code relative scatter columns ─────────────────
def _color_scatter(val):
    """Background colour: green < 1%, yellow 1-5%, orange 5-10%, red >10%."""
    if pd.isna(val):
        return ""
    if val < 1.0:
        return "background-color: #d4edda"  # green
    elif val < 5.0:
        return "background-color: #fff3cd"  # yellow
    elif val < 10.0:
        return "background-color: #ffd8b1"  # orange
    else:
        return "background-color: #f8d7da"  # red


scatter_pct_cols = ["RMS_%", "sigma_fit_%", "sigma_IQR_%"]
df_precision.style.applymap(_color_scatter, subset=scatter_pct_cols).format(
    {
        "median_flux_nJy": "{:.1f}",
        "RMS_nJy": "{:.1f}",
        "sigma_fit_nJy": "{:.1f}",
        "sigma_IQR_nJy": "{:.1f}",
        "median_err_nJy": "{:.1f}",
        "RMS_%": "{:.2f}",
        "sigma_fit_%": "{:.2f}",
        "sigma_IQR_%": "{:.2f}",
        "RMS_mmag": "{:.1f}",
        "sigma_fit_mmag": "{:.1f}",
        "sigma_IQR_mmag": "{:.1f}",
        "scatter/noise": "{:.2f}",
    }
)

In [ ]:
# ── Save precision table ──────────────────────────────────────────────
precision_out = os.path.join(DIR_FIGS, "photometric_precision_summary.csv")
df_precision.to_csv(precision_out, index=False)
log.info("Precision table saved: %s", precision_out)

# ── Print stars with excess scatter ──────────────────────────────────
excess = df_precision[df_precision["excess_scatter"]]
if len(excess):
    print(f"\n{len(excess)} (star x band) combinations with scatter/noise > 2:")
    display_cols = [
        "simbad_id",
        "band",
        "N",
        "sigma_IQR_nJy",
        "median_err_nJy",
        "scatter/noise",
        "sigma_IQR_%",
        "sigma_IQR_mmag",
    ]
    print(excess[display_cols].to_string(index=False))
else:
    print("No (star x band) combination with scatter/noise > 2 found.")

# ── Quick check: best band per star ──────────────────────────────────
print("\nBest (lowest sigma_IQR_%) band per star:")
best = df_precision.loc[
    df_precision.groupby("simbad_id")["sigma_IQR_%"].idxmin(),
    ["simbad_id", "band", "N", "sigma_IQR_%", "sigma_IQR_mmag"],
]
print(best.to_string(index=False))

## 9. Stacked normalised-flux residuals: 2×3 panel figure

For each of the 6 LSST bands (u g r i z y) arranged in a 2-row × 3-column
grid, this cell shows the **stacked histogram** of the normalised flux
residuals:

$$\delta_b = \frac{f - \tilde{f}_b}{\tilde{f}_b}$$

where $\tilde{f}_b$ is the per-star median flux in band $b$.  
Each star contributes a different colour; the global stacked distribution is
fitted by a Gaussian whose $\sigma_{\rm fit}$ is annotated.  
A shared legend listing all star names is placed below the 6 subplots.

Reading guide:
- $\sigma_{\rm fit} \lesssim 1\%$ → photometric precision better than 1%  
- $\sigma_{\rm fit} \approx 5\%$ → calibration scatter at the few-percent level  
- $\sigma_{\rm fit} \gtrsim 10\%$ → significant uncorrected systematics

In [ ]:
# ── Collect per-star residual arrays for each band ───────────────────────
# We use df_all loaded in Section 8 (already cleaned).

# Assign one colour per unique star (cycle through matplotlib tab10/tab20)
all_stars = sorted(df_all["simbad_id"].unique())
cmap = plt.get_cmap("tab10" if len(all_stars) <= 10 else "tab20")
star_colors = {sid: cmap(i % cmap.N) for i, sid in enumerate(all_stars)}

# Number of histogram bins for the residual distribution
N_BINS_RESID = 40

# x-axis range in fraction of median (e.g. ±20%)
RESID_XLIM = 0.20  # 20%

# ── Build figure with GridSpec ──────────────────────────────────────────
# Layout:
#   row 0-1  : 2 rows × 3 cols of band subplots
#   row 2    : legend row (short height)

fig_res = plt.figure(figsize=(15, 9))
fig_res.suptitle(
    r"Normalised flux residuals  $\delta = (f - \tilde{f}) / \tilde{f}$  per band"
    "\nStacked over all stable stars  —  colour per star  —  global Gaussian fit",
    fontsize=13,
    fontweight="bold",
    y=0.98,
)

gs_res = gridspec.GridSpec(
    3,  # 2 band rows + 1 legend row
    3,  # 3 columns
    figure=fig_res,
    hspace=0.52,
    wspace=0.32,
    height_ratios=[1, 1, 0.20],  # legend row is narrow
)

# Map band order to (row, col) positions in the 2×3 grid
band_pos = {
    "u": (0, 0),
    "g": (0, 1),
    "r": (0, 2),
    "i": (1, 0),
    "z": (1, 1),
    "y": (1, 2),
}

legend_handles = []  # collect for the shared legend below
legend_labels = []

for band in BAND_ORDER:
    row, col = band_pos[band]
    band_color = BAND_COLORS.get(band, "steelblue")
    ax = fig_res.add_subplot(gs_res[row, col])

    # ── Collect residuals for each star in this band ─────────────────
    all_resid = []  # all residuals stacked (for the global Gaussian fit)

    for star_idx, sid in enumerate(all_stars):
        mask = (df_all["simbad_id"] == sid) & (df_all[BAND_COL] == band) & df_all[FLUX_COL].notna()
        if FLUX_ERR_COL in df_all.columns:
            mask &= df_all[FLUX_ERR_COL].notna() & (df_all[FLUX_ERR_COL] > 0)

        sub = df_all.loc[mask, FLUX_COL].values
        if len(sub) < MIN_POINTS:
            continue

        med_star = np.nanmedian(sub)
        if abs(med_star) < 1e-12:
            continue

        resid = (sub - med_star) / abs(med_star)  # fractional residual
        all_resid.append(resid)

        # Per-star histogram (stacked = stacked bars)
        scolor = star_colors[sid]
        n_star, bins_star, patches = ax.hist(
            resid,
            bins=N_BINS_RESID,
            range=(-RESID_XLIM, RESID_XLIM),
            alpha=0.55,
            color=scolor,
            histtype="stepfilled",
            edgecolor="none",
            label=sid,
            stacked=False,  # overlapping (not stacked bars) for clarity
        )

        # Collect legend handles only once (from the first band)
        if band == BAND_ORDER[0]:
            legend_handles.append(patches[0])
            legend_labels.append(sid)

    # ── Global Gaussian fit over all stacked residuals ────────────────
    if all_resid:
        all_resid_cat = np.concatenate(all_resid)

        # Build the total histogram for the fit
        total_counts, total_edges = np.histogram(
            all_resid_cat,
            bins=N_BINS_RESID,
            range=(-RESID_XLIM, RESID_XLIM),
        )
        total_centers = 0.5 * (total_edges[:-1] + total_edges[1:])

        sig0 = sigma_iqr(all_resid_cat)
        sig0 = sig0 if sig0 > 0 else np.nanstd(all_resid_cat) or 0.01
        p0 = [0.0, sig0, total_counts.max()]

        try:
            popt, _ = curve_fit(
                gaussian,
                total_centers,
                total_counts,
                p0=p0,
                maxfev=5000,
            )
            mu_g, sig_g, amp_g = popt
            sig_g = abs(sig_g)

            if sig_g > 0 and abs(mu_g) < 5 * sig0:
                x_g = np.linspace(-RESID_XLIM, RESID_XLIM, 400)
                y_g = gaussian(x_g, mu_g, sig_g, amp_g)
                ax.plot(x_g, y_g, color="k", lw=2.0, ls="-", zorder=5, label="_nolegend_")
                ax.axvline(mu_g, color="k", lw=1.2, ls="--", zorder=5)

                # Annotate with sigma in % and mmag
                sig_pct = sig_g * 100.0
                sig_mmag = 1000.0 * (2.5 / np.log(10)) * sig_g
                ax.text(
                    0.97,
                    0.97,
                    f"$\\sigma_{{\\rm fit}}$ = {sig_pct:.2f}%\n= {sig_mmag:.1f} mmag",
                    transform=ax.transAxes,
                    fontsize=9,
                    va="top",
                    ha="right",
                    color="k",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.8),
                    zorder=6,
                )
        except (RuntimeError, ValueError):
            ax.text(
                0.97,
                0.97,
                "fit failed",
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                ha="right",
                color="red",
            )

    # ── Reference vertical lines at ±1%, ±5%, ±10% ───────────────────
    for ref_pct, ls_, alpha_ in [(0.01, ":", 0.5), (0.05, "--", 0.4), (0.10, "-.", 0.3)]:
        for sign in (-1, 1):
            ax.axvline(sign * ref_pct, color="grey", lw=0.8, ls=ls_, alpha=alpha_)

    ax.set_xlim(-RESID_XLIM, RESID_XLIM)
    ax.set_xlabel(r"$(f - \tilde{f}) / \tilde{f}$", fontsize=9)
    ax.set_ylabel("counts", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, lw=0.3, alpha=0.35)

    # Band label as title with band colour
    ax.set_title(f"{band} band", fontsize=11, color=band_color, fontweight="bold")

    # x-tick labels as percentages
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1.0, decimals=0))

# ── Shared legend placed in the 3rd (legend) row spanning all columns ─────
# Use a dedicated invisible axes for the legend
ax_leg = fig_res.add_subplot(gs_res[2, :])
ax_leg.axis("off")

if legend_handles:
    leg = ax_leg.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=min(len(legend_labels), 6),
        fontsize=9,
        frameon=True,
        framealpha=0.85,
        title="Stable stars (Simbad IDs)",
        title_fontsize=9,
        handlelength=1.5,
        handleheight=0.9,
        borderpad=0.6,
        columnspacing=1.0,
    )

savefig(fig_res, "stacked_normalised_flux_residuals_2x3")
plt.show()
log.info("Stacked residuals figure saved.")

In [ ]:
# ── Quick summary: sigma_fit of the global residual per band ─────────────
print("Global photometric precision (sigma_fit of stacked residuals):")
print(f"{'Band':>6}  {'sigma_fit_%':>12}  {'sigma_fit_mmag':>15}  {'N_total':>8}")
print("-" * 48)

for band in BAND_ORDER:
    mask_b = (df_all[BAND_COL] == band) & df_all[FLUX_COL].notna()
    if FLUX_ERR_COL in df_all.columns:
        mask_b &= df_all[FLUX_ERR_COL].notna() & (df_all[FLUX_ERR_COL] > 0)

    all_band_resid = []
    for sid in all_stars:
        sub = df_all.loc[mask_b & (df_all["simbad_id"] == sid), FLUX_COL].values
        if len(sub) < MIN_POINTS:
            continue
        med_s = np.nanmedian(sub)
        if abs(med_s) < 1e-12:
            continue
        all_band_resid.append((sub - med_s) / abs(med_s))

    if not all_band_resid:
        print(f"{band:>6}  {'--':>12}  {'--':>15}  {'--':>8}")
        continue

    all_r = np.concatenate(all_band_resid)
    sig0 = sigma_iqr(all_r)
    sig0 = sig0 if sig0 > 0 else 0.01

    counts_b, edges_b = np.histogram(all_r, bins=N_BINS_RESID, range=(-RESID_XLIM, RESID_XLIM))
    centers_b = 0.5 * (edges_b[:-1] + edges_b[1:])

    try:
        popt_b, _ = curve_fit(
            gaussian,
            centers_b,
            counts_b,
            p0=[0.0, sig0, counts_b.max()],
            maxfev=5000,
        )
        sig_b = abs(popt_b[1])
        sig_pct_b = sig_b * 100.0
        sig_mmag_b = 1000.0 * (2.5 / np.log(10)) * sig_b
        print(f"{band:>6}  {sig_pct_b:>12.3f}  {sig_mmag_b:>15.1f}  {len(all_r):>8}")
    except (RuntimeError, ValueError):
        print(f"{band:>6}  {'fit failed':>12}  {'--':>15}  {len(all_r):>8}")